# Binary Classification with `sltn`

This notebook demonstrates how to use `sltn` for a simple binary classification task formulated in first-order logic and trained with a neural interpretation.

## Problem

We sample 100 two-dimensional points from the unit square. A point is labelled positive if it lies inside the disc of radius $0.3$ centered at $(0.5, 0.5)$, and negative otherwise.

The goal is to learn a unary predicate $A$ over points such that:

$$
\mathcal{K}
=
\left\{
\forall x_{\mathrm{pos}}\, A(x_{\mathrm{pos}}),
\quad
\forall x_{\mathrm{neg}}\, \lnot A(x_{\mathrm{neg}})
\right\}.
$$

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset

We generate a synthetic two-dimensional dataset and assign labels using the known geometric rule.

A point x = (x_1, x_2) is positive when:

$$
\|x - (0.5, 0.5)\|_2^2 < 0.3^2.
$$

The plot below shows the ground-truth partition of the sampled points.

In [ ]:
import matplotlib.pyplot as plt

nr_samples = 100
dataset = torch.rand((nr_samples, 2))
labels_dataset = torch.sum(torch.square(dataset - torch.tensor([.5, .5])), dim=1) < .09

plt.figure(figsize=(4, 4))
plt.scatter(dataset[labels_dataset][:, 0], dataset[labels_dataset][:, 1], label='A')
plt.scatter(dataset[~labels_dataset][:, 0], dataset[~labels_dataset][:, 1], label='~A')
plt.title('Ground truth')
plt.legend()
plt.show()

## 1. Signature: declaring the logical vocabulary

A `Signature` defines the symbolic vocabulary used to write logical formulas.

In this example, the signature contains:

- a sort `Point`, representing two-dimensional input points;
- variables `x_pos`, `x_neg`, and `x`, all of sort `Point`;
- a unary predicate `A`, which maps a `Point` to a truth value.

The signature only declares symbols and their types. It does not assign tensors, neural networks, or concrete values to them. Those assignments are provided later through an `Interpretation`.

In [ ]:
import sltn
from sltn import Signature, Interpretation, KB, Real

# Declare signature
sig = Signature("binary_clf")
sig.sort("Point")
sig.variable("x_pos", "Point")
sig.variable("x_neg", "Point")
sig.variable("x", "Point")
sig.predicate("A", ["Point"])

print(sig)

## 2. Formulas and knowledge base

Formulas are constructed from the symbols declared in the signature using first-order syntax.

Here we define two universally quantified constraints:

- every positive example should satisfy $A$;
- every negative example should satisfy $\lnot A$.

The `KB` object groups these formulas into a single knowledge base whose satisfaction can be evaluated under an interpretation.

In [ ]:
phi_pos = sig.parse("forall x_pos: A(x_pos)")
phi_neg = sig.parse("forall x_neg: not A(x_neg)")
phi_kb  = KB([phi_pos, phi_neg])

print("phi_pos :", phi_pos)
print("phi_neg :", phi_neg)
print("phi_kb  :", phi_kb)

## 3. Neural classifier

The predicate `A` will be grounded by a neural network.

The model receives a two-dimensional point and returns a scalar value in $[0, 1]$. The final sigmoid activation makes the output suitable for interpretation as a fuzzy truth value:
values close to $1$ indicate strong evidence for $A(x)$, while values close to $0$ indicate strong evidence for $\lnot A(x)$.

In [ ]:
class ModelA(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 1), torch.nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## 4. Interpretation: grounding the vocabulary

An `Interpretation` maps the symbolic objects declared in the signature to concrete computational objects.

In this notebook:

- the sort `Point` is interpreted as a two-dimensional real-valued tensor type;
- the predicate `A` is interpreted as an instance of `ModelA`;
- the variables `x_pos`, `x_neg`, and `x` are assigned tensors during evaluation or training.

Once the interpretation is defined, formulas can be evaluated as differentiable expressions whose truth values depend on the neural predicate.

In [ ]:
# Declare interpretation
interp = Interpretation(sig).to(device)
interp["Point"] = sltn.Type("Point", shape=(2,))  # 2-D real vectors
interp["A"] = ModelA()

## 5. Data loaders and metrics

Here follow a standard dataloader, and metrics computation utility. The satisfaction level measures how well the interpreted knowledge base is satisfied. Accuracy is computed by thresholding the predicate value $A(x)$ at $0.5$.

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

class DataLoader:
    def __init__(self, data, labels, batch_size=64, shuffle=True):
        self.data = data.to(device)
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        n = self.data.shape[0]
        idxlist = list(range(n))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, n, self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.data[idx], self.labels[idx]


def compute_sat_level(loader, interp):
    mean_sat = 0
    for data, labels in loader:
        # Bind per-batch slices via .I and let interp resolve them automatically
        interp["x_pos"] = data[labels]
        interp["x_neg"] = data[~labels]
        mean_sat += interp(phi_kb).value
    return mean_sat / len(loader)


def compute_accuracy(loader, interp):
    mean_acc = 0.0
    for data, labels in loader:
        with torch.no_grad():
            preds = interp["A"](data).cpu().numpy()
        preds = np.where(preds > 0.5, 1., 0.).flatten()
        mean_acc += accuracy_score(labels, preds)
    return mean_acc / len(loader)


nr_train = nr_samples * 75 // 100
train_loader = DataLoader(dataset[:nr_train], labels_dataset[:nr_train], batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset[nr_train:], labels_dataset[nr_train:], batch_size=nr_samples-nr_train, shuffle=False)

## 6. Training loop

Training maximizes the satisfaction of the knowledge base.

For each batch:

1. positive and negative examples are assigned to `x_pos` and `x_neg`;
2. the knowledge base is evaluated under the current interpretation;
3. the loss is defined as: $\mathcal{L} = 1 - \mathrm{SatAgg}(\mathcal{K}).$

In [ ]:
optimizer = torch.optim.Adam(interp["A"].parameters(), lr=0.001)

for epoch in range(1000):
    train_loss = 0.0
    for data, labels in train_loader:
        optimizer.zero_grad()
        interp["x_pos"], interp["x_neg"] = data[labels], data[~labels]
        loss = 1 - interp(phi_kb).value
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d} | loss {train_loss:.4f} "
              f"| Train Sat {compute_sat_level(train_loader, interp).item():.3f} "
              f"| Test Sat {compute_sat_level(test_loader, interp).item():.3f} "
              f"| Train Acc {compute_accuracy(train_loader, interp):.3f} "
              f"| Test Acc {compute_accuracy(test_loader, interp):.3f}")

## 7. Visualization

Finally, we inspect the learned predicate on the training and test splits. 
The plots show the fuzzy truth values assigned by the model to:
the learned positive-class predicate $A(x)$ and its logical negation.

In [ ]:
nr_train = nr_samples * 75 // 100
fig, axes = plt.subplots(2, 2, figsize=(9, 9))

for row, (split, data_split) in enumerate([
        ("Training", dataset[:nr_train]),
        ("Test",     dataset[nr_train:])]):

    interp["x"] = data_split

    with torch.no_grad():
        v_A    = interp("A(x)").value.cpu().numpy().squeeze()
        v_notA = interp("not A(x)").value.cpu().numpy().squeeze()

    axes[row, 0].scatter(data_split[:, 0], data_split[:, 1], c=v_A, vmin=0, vmax=1)
    axes[row, 0].set_title(f"A(x) — {split}")
    axes[row, 1].scatter(data_split[:, 0], data_split[:, 1], c=v_notA, vmin=0, vmax=1)
    axes[row, 1].set_title(f"not A(x) — {split}")

plt.tight_layout()
plt.show()